# Tutorial: Define molecular reward
- Define a reward for molecular generation
- Generation with custom reward

## Define a reward for molecular generation
A reward class maps nodes to real values, representing the objective to be optimized. We also support multi-objective settings: although the objectives must ultimately be folded into a single scalar value for optimization, our method allows tracking of individual objective values throughout the process.

For molecular generation, we recommend implementing a reward by inheriting from `MolReward`. For non-molecular nodes, see `2a_advanced_make_nonmol_components.ipynb`.

In [ ]:
# Define a molecular reward

import numpy as np
from rdkit.Chem import Descriptors
from chemtsv3.reward import MolReward

def sigmoid(x, a):
    return 1 / (1 + np.exp(-a * x))

class ExampleReward(MolReward):
    """Reward based on (LogP value - max ring size)."""
    def __init__(self, a):
        self.a = a
        
    def mol_objective_functions(self):
        """Return objective functions of the node; each function returns an objective value."""
        
        def log_p(mol):
            return Descriptors.MolLogP(mol)
        
        def max_ring_size(mol):
            ri = mol.GetRingInfo()
            max_ring_size = max((len(r) for r in ri.AtomRings()), default=0)
            return max_ring_size

        return [log_p, max_ring_size]
    

    def reward_from_objective_values(self, objective_values):
        """Compute the final reward based on the objective values calculated by objective_functions()."""
        log_p, max_ring_size = objective_values[0], objective_values[1]
        return sigmoid(x=log_p - max_ring_size, a=self.a) # It is recommended to scale the reward to the range [0, 1].

## Generation with custom reward
For YAML workflow, define the reward in a `***.py` file and place it in the `reward` directory. (You can find the same `ExampleReward` as above in `reward/example_reward.py`.)

In YAML file, you can specify the reward class and its `__init__` arguments like this (see `config/tutorial/1b_example.yaml`.):
```
reward_class: ExampleReward
reward_args:
  a: 0.5
```

In [ ]:
import os
from chemtsv3.utils import conf_from_yaml, generator_from_conf

# Generate molecules with ExampleReward

repo_root = "../../"
yaml_path = "config/tutorial/1b_example.yaml" # Specify the yaml path. Open the file to see setting options.
yaml_path = os.path.join(repo_root, yaml_path)

conf = conf_from_yaml(yaml_path)
generator = generator_from_conf(conf, base_dir=repo_root)
generator.generate(time_limit=conf.get("time_limit"), max_generations=conf.get("max_generations"))
generator.plot(**conf.get("plot_args"))